In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import pytz
import re

In [2]:
app_data = pd.read_csv('..//Data//Cleaned_Data//eda_prepared_dataset.csv')
app_data.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,...,Updated_Year,Avg_Sentiment_Polarity,Avg_Sentiment_Subjectivity,Total_Reviews,Avg_Review_Length,Avg_Word_Count,Revenue,Update_Month_Name,Update_Year,Year_Month
0,GollerCepte Live Score,SPORTS,4.2,9992,31.000000,1000000,Free,0.0,Everyone,Sports,...,2018,0.000000,0.000000,0.0,0.000000,0.000000,0.0,May,2018,2018-05
1,Ad Block REMOVER - NEED ROOT,TOOLS,3.3,999,0.088867,100000,Free,0.0,Everyone,Tools,...,2013,0.000000,0.000000,0.0,0.000000,0.000000,0.0,December,2013,2013-12
2,SnipSnap Coupon App,SHOPPING,4.2,9975,18.000000,1000000,Free,0.0,Everyone,Shopping,...,2018,0.000000,0.000000,0.0,0.000000,0.000000,0.0,January,2018,2018-01
3,US Open Tennis Championships 2018,SPORTS,4.0,9971,33.000000,1000000,Free,0.0,Everyone,Sports,...,2018,0.000000,0.000000,0.0,0.000000,0.000000,0.0,June,2018,2018-06
4,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.000000,500000,Free,0.0,Teen,Travel & Local,...,2018,0.371277,0.438569,34.0,71.088235,10.941176,0.0,August,2018,2018-08


In [3]:
review_data = pd.read_csv('..//Data//Cleaned_Data//review_sentiment_eda_dataset.csv')
review_data.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,...,Updated_Year,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Review_length,Word_count,Sentiment_Label,Subjectivity_Category,Polarity_Type
0,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.0,500000,Free,0.0,Teen,Travel & Local,...,2018,I'm grateful booking engine dreamtrips. I've a...,Positive,0.2000,0.2000,150.0,21.0,1.0,Low,Neutral
1,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.0,500000,Free,0.0,Teen,Travel & Local,...,2018,"This great app. I One book plane, hotel, dream...",Positive,0.8500,0.8750,125.0,20.0,1.0,High,Neutral
2,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.0,500000,Free,0.0,Teen,Travel & Local,...,2018,Its slow rovia keeps going back disappointed g...,Negative,-0.1375,0.4125,58.0,10.0,0.0,Medium,Positive
3,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.0,500000,Free,0.0,Teen,Travel & Local,...,2018,Try get dreams trips Germany UK,Neutral,0.0000,0.0000,31.0,6.0,-1.0,Low,Neutral
4,DreamTrips,TRAVEL_AND_LOCAL,4.7,9971,22.0,500000,Free,0.0,Teen,Travel & Local,...,2018,I cant open travelbook samsung handphone,Neutral,0.0000,0.5000,40.0,6.0,-1.0,Medium,Neutral


## Helper Functions

### IST Time Visibility Functions

In [4]:
# IST time function
def get_ist_hour():

    india_tz = pytz.timezone("Asia/Kolkata")

    current_time = datetime.now(india_tz)

    return current_time.hour

In [5]:
# Time Range Checker Function
def is_time_between(start_hour, end_hour):

    current_hour = get_ist_hour()

    return start_hour <= current_hour < end_hour

### Number Formatter Function

In [6]:
# Human Readable Number Formatter
def human_format(num):
    if num >= 1e12:
        return f"{num/1e12:.1f}T"

    elif num >= 1e9:
        return f"{num/1e9:.1f}B"

    elif num >= 1e6:
        return f"{num/1e6:.1f}M"

    elif num >= 1e3:
        return f"{num/1e3:.1f}K"

    return str(num)

In [7]:
# Validation of human_format function
print(human_format(1500))
print(human_format(2500000))
print(human_format(7000000000))

1.5K
2.5M
7.0B


## App Name Validation Function

In [8]:
# Check App Name Length
def valid_app_length(app_name):

    return len(str(app_name)) <= 30

In [9]:
print(valid_app_length("Instagram"))
print(valid_app_length("This is a very very long app name"))

True
False


In [10]:
# Check If App Name Contains Letter "S"
def contains_s(app_name):

    return "s" in str(app_name).lower()

In [11]:
print(contains_s("WhatsApp"))
print(contains_s("Telegram"))

True
False


In [12]:
# Check If App Starts with X/Y/Z
def starts_with_xyz(app_name):

    app_name = str(app_name).lower()

    return app_name.startswith(
        ("x", "y", "z")
    )

In [13]:
print(starts_with_xyz("Xender"))
print(starts_with_xyz("YouTube"))
print(starts_with_xyz("Instagram"))

True
True
False


In [14]:
# Check If App Name Contains Numbers
def contains_number(app_name):

    return bool(
        re.search(r"\d", str(app_name))
    )

In [15]:
print(contains_number("PUBG"))
print(contains_number("PUBG123"))

False
True


## Task-wise Filtered Data Preparation

### Task-1

In [16]:
# Filter January Apps
task1_df = app_data[
    app_data["Update_Month_Name"] == "January"
]

In [17]:
# Filter Rating > 4
task1_df = task1_df[
    task1_df["Rating"] > 4.0
]

In [18]:
# Filter Size > 10 MB
task1_df = task1_df[
    task1_df["Size"] > 10
]

In [19]:
# Group by Category
task1_grouped = (
    task1_df.groupby("Category")
    .agg({
        "Rating": "mean",
        "Reviews": "sum",
        "Installs": "sum"
    })
    .reset_index()
)

In [20]:
# Sort by Installs
task1_grouped = (
    task1_grouped
    .sort_values(
        by="Installs",
        ascending=False
    )
)

In [21]:
# Keep Top 10 Categories
task1_grouped = task1_grouped.head(10)

In [22]:
# Preview Dataset
task1_grouped

,Category,Rating,Reviews,Installs
22,SPORTS,4.400000,1988557,120621000
11,GAME,4.342857,2209062,108691000
8,FAMILY,4.432558,3056570,108595820
23,TOOLS,4.285714,331560,72010000
17,PERSONALIZATION,4.475000,155996,15060000
19,PRODUCTIVITY,4.440000,221779,11610000
3,COMMUNICATION,4.300000,367412,11501000
6,ENTERTAINMENT,4.350000,745832,11000000
18,PHOTOGRAPHY,4.300000,542561,10000000
24,TRAVEL_AND_LOCAL,4.100000,974,1001000


In [23]:
task1_grouped.to_csv("../Data/Task_wise_dataset/task1_grouped.csv",index=False)

### Task-2

In [24]:
# Remove Categories Starting with A/C/G/S
task2_df = app_data[
    ~app_data["Category"]
    .str.startswith(("A", "C", "G", "S"))
]

In [25]:
# Filter Installs Greater than 1 Million
task2_df = task2_df[
    task2_df["Installs"] > 1_000_000
]

In [26]:
# Group by Category
task2_grouped = (
    task2_df.groupby("Category")
    .agg({
        "Installs": "sum"
    })
    .reset_index()
)

In [27]:
# Sort by Installs
task2_grouped = (
    task2_grouped
    .sort_values(
        by="Installs",
        ascending=False
    )
)

In [28]:
# Keep Top 5 Categories
task2_grouped = task2_grouped.head(5)

In [29]:
# Preview Dataset
task2_grouped

,Category,Installs
21,TOOLS,7875000000
7,FAMILY,5875000000
20,PRODUCTIVITY,5725000000
19,PHOTOGRAPHY,4590000000
23,VIDEO_PLAYERS,3895000000


In [30]:
task2_grouped.to_csv("../Data/Task_wise_dataset/task2_grouped.csv",index=False)

### Task-3

In [31]:
# Filter Installs
task3_df = app_data[
    app_data["Installs"] > 10_000
]

In [32]:
# Filter Revenue
task3_df = task3_df[
    task3_df["Revenue"] > 10_000
]

In [33]:
# Filter Size > 15 MB
task3_df = task3_df[
    task3_df["Size"] > 15
]

In [34]:
# Filter Content Rating
task3_df = task3_df[
    task3_df["Content Rating"] == "Everyone"
]

In [35]:
# Extract Android Version Number
task3_df["Android_Version_Num"] = (
    task3_df["Android Ver"]
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)

In [36]:
# Filter App Name Length
task3_df = task3_df[
    task3_df["App"].apply(valid_app_length)
]

In [37]:
# Find Top 3 Categories
top3_categories = (
    task3_df.groupby("Category")["Installs"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
)

top3_categories

Index(['FAMILY', 'GAME', 'PERSONALIZATION'], dtype='str', name='Category')

In [38]:
# Keep Only Top 3 Categories
task3_df = task3_df[
    task3_df["Category"].isin(top3_categories)
]

In [39]:
# Group Data for Visualization
task3_grouped = (
    task3_df.groupby(
        ["Category", "Type"]
    )
    .agg({
        "Installs": "mean",
        "Revenue": "mean"
    })
    .reset_index()
)

In [40]:
# Preview Dataset
task3_grouped

,Category,Type,Installs,Revenue
0,FAMILY,Paid,320000.0,936800.000000
1,GAME,Paid,237500.0,426791.666667
2,PERSONALIZATION,Paid,525000.0,557250.000000


In [41]:
task3_grouped.to_csv("../Data/Task_wise_dataset/task3_grouped.csv",index=False)

### Task-4

In [42]:
# Filter Categories Starting with E/C/B
task4_df = app_data[
    app_data["Category"]
    .str.startswith(("E", "C", "B"))
]

In [43]:
# Filter Reviews > 500
task4_df = task4_df[
    task4_df["Reviews"] > 500
]

In [44]:
# Remove Apps Starting with X/Y/Z
task4_df = task4_df[
    ~task4_df["App"]
    .apply(starts_with_xyz)
]

In [45]:
# Remove Apps Containing "S"
task4_df = task4_df[
    ~task4_df["App"]
    .apply(contains_s)
]

In [ ]:
# Create Last Updated Column
task4_df["Last Updated"] = pd.to_datetime(
    task4_df["Last Updated"],
    errors="coerce"
)

In [47]:
# Create Year-Month Column
task4_df["Year_Month"] = (
    task4_df["Last Updated"]
    .dt.to_period("M")
    .astype(str)
)

In [48]:
# Group Data by Time & Category
task4_grouped = (
    task4_df.groupby(
        ["Year_Month", "Category"]
    )
    .agg({
        "Installs": "sum"
    })
    .reset_index()
)

In [49]:
# Sort Data
task4_grouped = task4_grouped.sort_values(
    by=["Category", "Year_Month"]
)

In [50]:
# Calculate MoM Growth
task4_grouped["MoM_Growth"] = (
    task4_grouped.groupby("Category")["Installs"]
    .pct_change() * 100
)

In [51]:
# Preview Dataset
task4_grouped.head(15)

,Year_Month,Category,Installs,MoM_Growth
46,2018-03,BEAUTY,5000,NaN
58,2018-06,BEAUTY,1000000,19900.000000
64,2018-07,BEAUTY,1000000,0.000000
72,2018-08,BEAUTY,100000,-90.000000
5,2014-10,BOOKS_AND_REFERENCE,500000,NaN
6,2014-11,BOOKS_AND_REFERENCE,5000000,900.000000
10,2015-07,BOOKS_AND_REFERENCE,10000000,100.000000
18,2016-06,BOOKS_AND_REFERENCE,60000,-99.400000
21,2016-08,BOOKS_AND_REFERENCE,100000,66.666667
27,2017-02,BOOKS_AND_REFERENCE,100000,0.000000


In [52]:
task4_grouped.to_csv("../Data/Task_wise_dataset/task4_grouped.csv",index=False)

### Task-5

In [53]:
# Define Allowed Categories
allowed_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

In [54]:
# Filter Categories
task5_df = app_data[
    app_data["Category"]
    .isin(allowed_categories)
]

In [55]:
# Filter Rating > 3.5
task5_df = task5_df[
    task5_df["Rating"] > 3.5
]

In [56]:
# Filter Reviews > 500
task5_df = task5_df[
    task5_df["Reviews"] > 500
]

In [57]:
# Filter Installs > 50K
task5_df = task5_df[
    task5_df["Installs"] > 50_000
]

In [58]:
# Remove Apps Containing "S"
task5_df = task5_df[
    ~task5_df["App"]
    .apply(contains_s)
]

In [59]:
# Filter Subjectivity > 0.5
task5_df = task5_df[
    task5_df["Avg_Sentiment_Subjectivity"] > 0.5
]

In [60]:
# Preview Dataset
task5_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,...,Updated_Year,Avg_Sentiment_Polarity,Avg_Sentiment_Subjectivity,Total_Reviews,Avg_Review_Length,Avg_Word_Count,Revenue,Update_Month_Name,Update_Year,Year_Month
145,Caller ID +,COMMUNICATION,4.0,9498,0.115234,1000000,Free,0.0,Everyone,Communication,...,2016,0.350000,0.600000,1.0,97.000000,15.000000,0.0,June,2016,2016-06
354,Hill Climb Racing,GAME,4.4,8923847,63.000000,100000000,Free,0.0,Everyone,Racing,...,2018,0.106515,0.508071,36.0,158.444444,26.416667,0.0,July,2018,2018-07
1548,Chrome Dev,COMMUNICATION,4.4,63576,13.000000,5000000,Free,0.0,Everyone,Communication,...,2018,0.305890,0.505686,37.0,121.891892,17.297297,0.0,August,2018,2018-08
1615,Google Primer,BUSINESS,4.4,62272,18.000000,10000000,Free,0.0,Everyone,Business,...,2018,0.750000,0.675000,2.0,4.500000,1.000000,0.0,June,2018,2018-06
1800,Block Puzzle,GAME,4.6,59907,7.800000,5000000,Free,0.0,Everyone,Puzzle,...,2018,0.049265,0.536908,93.0,124.741935,20.258065,0.0,March,2018,2018-03


In [62]:
task5_df.to_csv("../Data/Task_wise_dataset/task5_grouped.csv",index=False)

### Task-6

In [63]:
# Filter Categories Starting with T or P
task6_df = app_data[
    app_data["Category"]
    .str.startswith(("T", "P"))
]

In [65]:
# Filter Rating >= 4.2
task6_df = task6_df[
    task6_df["Rating"] >= 4.2
]

In [66]:
# Filter Reviews > 1000
task6_df = task6_df[
    task6_df["Reviews"] > 1000
]

In [67]:
# Filter Size Between 20 MB and 80 MB
task6_df = task6_df[
    (task6_df["Size"] >= 20)
    &
    (task6_df["Size"] <= 80)
]

In [68]:
# Remove Apps Containing Numbers
task6_df = task6_df[
    ~task6_df["App"]
    .apply(contains_number)
]

In [69]:
# Convert Last Updated to Datetime
task6_df["Last Updated"] = pd.to_datetime(
    task6_df["Last Updated"],
    errors="coerce"
)

In [70]:
# Create Year-Month Column
task6_df["Year_Month"] = (
    task6_df["Last Updated"]
    .dt.to_period("M")
    .astype(str)
)

In [71]:
# Group by Time and Category
task6_grouped = (
    task6_df.groupby(
        ["Year_Month", "Category"]
    )
    .agg({
        "Installs": "sum"
    })
    .reset_index()
)

In [73]:
# Sort Dataset
task6_grouped = task6_grouped.sort_values(
    by=["Category", "Year_Month"]
)

In [74]:
# Calculate MoM Growth
task6_grouped["MoM_Growth"] = (
    task6_grouped.groupby("Category")["Installs"]
    .pct_change() * 100
)

In [75]:
# Create Cumulative Installs
task6_grouped["Cumulative_Installs"] = (
    task6_grouped.groupby("Category")["Installs"]
    .cumsum()
)

In [77]:
# Preview Dataset
task6_grouped.head(15)

,Year_Month,Category,Installs,MoM_Growth,Cumulative_Installs
16,2018-03,PARENTING,100000,NaN,100000
19,2018-05,PARENTING,10000000,9900.000000,10100000
26,2018-07,PARENTING,600000,-94.000000,10700000
2,2016-12,PERSONALIZATION,1000000,NaN,1000000
7,2017-09,PERSONALIZATION,1000000,0.000000,2000000
11,2018-01,PERSONALIZATION,10000000,900.000000,12000000
14,2018-02,PERSONALIZATION,10000000,0.000000,22000000
22,2018-06,PERSONALIZATION,10000000,0.000000,32000000
27,2018-07,PERSONALIZATION,11500000,15.000000,43500000
32,2018-08,PERSONALIZATION,20000000,73.913043,63500000


In [78]:
task6_grouped.shape

(37, 5)

In [79]:
task6_grouped.to_csv("../Data/Task_wise_dataset/task6_grouped.csv",index=False)